## 🌐 Advanced Tool Use with LangGraph (Python)

Modeled on the companion notebook for the Microsoft Agent Framework, this version shows how to implement a multi-tool travel assistant using **LangGraph** (LangChain's graph-based orchestration layer).

### 📋 Learning Objectives
- Build a graph-based agent that can choose and execute tools
- Model agent state with a TypedDict
- Add and register structured LangChain tools
- Demonstrate sequential and (conceptually) parallel tool usage
- Compare good vs bad tool design patterns

### 🧩 Architecture Overview
The graph orchestrates: user -> model reasoning -> optional tool invocation -> response synthesis.

```mermaid
flowchart LR
  U[User Input] --> R[Router / Policy Node] -->|select tool| T[Tool Node] --> S[State Merge] --> M[Model Response] --> O[Output]
  R -->|no tool| M
```

### 🔑 Why LangGraph?
- Deterministic control over branching/tool paths
- Explicit state object (traceable & inspectable)
- Easy extension to multi-agent / parallel tools

### ⚙️ Prerequisites & Setup
Install (latest):
```bash
pip install -U langgraph langchain langchain-core langchain-openai openai
```
Environment variables (example):
```env
OPENAI_API_KEY=your_key_here
OPENAI_MODEL=gpt-4o-mini   # or gpt-4o, gpt-4.1 etc
OPENAI_BASE_URL=https://api.openai.com/v1  # or custom endpoint (GitHub Models gateway, Azure, etc)
```
If using Azure / GitHub Models proxy set the base URL + api key accordingly.

> NOTE: This notebook uses stubbed data for weather & cost; replace those with real API integrations as needed.

In [1]:
# Imports & basic setup
import os, math, json, asyncio
from typing import TypedDict, List, Dict, Any, Literal, Optional
from dotenv import load_dotenv
load_dotenv()

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_openai import AzureChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver  # simple in-memory persistence


In [2]:
azure_endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
azure_api_key = os.getenv("AZURE_AI_FOUNDRY_API_KEY")
# Support either AZURE_AI_FOUNDRY_MODEL_ID or legacy AZURE_AI_FOUNDRY_MODEL
azure_deployment = os.getenv("AZURE_AI_FOUNDRY_MODEL_ID") or os.getenv("AZURE_AI_FOUNDRY_MODEL")
api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-07-01-preview")

# Debug prints (mask key)
print(f"Azure Endpoint: {azure_endpoint}")
print(f"Azure Deployment: {azure_deployment}")
print(f"API Version: {api_version}")



Azure Endpoint: https://ibecfoundry.openai.azure.com/
Azure Deployment: gpt-4o
API Version: 2024-02-01


In [3]:
MODEL_ID = os.getenv('AZURE_AI_FOUNDRY_MODEL','gpt-4o')
# Initialize Azure OpenAI chat model
llm = AzureChatOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version=api_version,
    deployment_name=azure_deployment,
    temperature=0.7
)
print('Model configured:', MODEL_ID)

Model configured: gpt-4o


### 🔨 Defining Tools (Good Examples)
Each tool: clear name, validated inputs, typed output, single responsibility.
Return structured JSON-friendly data where downstream reasoning might inspect fields.

In [4]:
from random import randint

DESTINATIONS = ["Paris, France", "Tokyo, Japan", "Barcelona, Spain", "Reykjavik, Iceland", "Cape Town, South Africa", "Bali, Indonesia"]

@tool
def get_random_destination() -> str:
    """Return a random travel destination (inspiration tool).
    Returns: destination (str)
    Edge Cases: None (list non-empty)
    """
    return DESTINATIONS[randint(0, len(DESTINATIONS)-1)]

@tool
def get_weather_info(location: str) -> dict:
    """Stub weather retrieval for a location.
    Args: location (non-empty)
    Returns: {location, forecast, temp_c}
    """
    if not location or not location.strip():
        raise ValueError('location cannot be empty')
    base = sum(ord(c) for c in location.lower()) % 3
    forecast = ['sunny','cloudy','showers'][base]
    temp_c = 16 + base * 3.5
    return {'location': location, 'forecast': forecast, 'temp_c': temp_c}

@tool
def calculate_travel_cost(destination: str, days: int, style: Literal['budget','standard','luxury']='standard') -> dict:
    """Estimate travel cost based on style & duration.
    Returns breakdown dict.
    """
    if not destination or not destination.strip():
        raise ValueError('destination cannot be empty')
    if days <= 0:
        raise ValueError('days must be > 0')
    daily = {'budget':120,'standard':230,'luxury':480}[style]
    return {
        'destination': destination,
        'days': days,
        'style': style,
        'daily_rate': daily,
        'estimate_total': daily * days
    }

@tool
def recommend_local_activities(destination: str, interests: List[str], max_results: int = 4) -> list:
    """Recommend local activities matching interests.
    Returns list[{name, category, destination}]
    """
    if not destination.strip():
        raise ValueError('destination required')
    if not interests:
        raise ValueError('at least one interest required')
    catalog = {
        'art':['Gallery tour','Modern art museum','Street mural walk'],
        'food':['Food market crawl','Regional tasting','Cooking class'],
        'outdoors':['Coastal hike','City bike loop','Botanical garden'],
        'relax':['Spa session','Sunset lounge','Mindfulness retreat']
    }
    chosen_cats = []
    for interest in interests:
        low = interest.lower()
        if any(k in low for k in ['paint','art','gallery','museum']): chosen_cats.append('art')
        elif any(k in low for k in ['food','eat','taste','cuisine']): chosen_cats.append('food')
        elif any(k in low for k in ['hike','bike','trail','outdoor']): chosen_cats.append('outdoors')
        elif any(k in low for k in ['relax','spa','calm','zen']): chosen_cats.append('relax')
    if not chosen_cats: chosen_cats.append('art')
    seen, out = set(), []
    for cat in chosen_cats:
        if cat in seen: continue
        seen.add(cat)
        for act in catalog.get(cat, []):
            out.append({'name': act, 'category': cat, 'destination': destination})
            if len(out) >= max_results: return out
    return out

TOOLS = [get_random_destination, get_weather_info, calculate_travel_cost, recommend_local_activities]
print('Registered tools:', [t.name for t in TOOLS])

Registered tools: ['get_random_destination', 'get_weather_info', 'calculate_travel_cost', 'recommend_local_activities']


### 🧪 Agent State Definition
We maintain: conversation (list of messages), last_tool (name), and an aggregated info field for tool outputs.

In [5]:
class AgentState(TypedDict, total=False):
    messages: List
    aggregated: Dict[str, Any]
    last_tool: Optional[str]

def ensure_state(state: AgentState) -> AgentState:
    state.setdefault('messages', [])
    state.setdefault('aggregated', {})
    return state

ensure_state({})

{'messages': [], 'aggregated': {}}

### 🧠 Policy / Router Node
Decides whether to call a tool or respond directly. Simple heuristic: if user mentions keywords (weather, cost, activities, random) -> tool path.

In [6]:
def router_node(state: AgentState):
    state = ensure_state(state)
    last_msg = state['messages'][-1] if state['messages'] else None
    if isinstance(last_msg, HumanMessage):
        text = last_msg.content.lower()
        if 'weather' in text: return {'tool_to_call': 'get_weather_info'}
        if 'cost' in text or 'price' in text or 'budget' in text: return {'tool_to_call': 'calculate_travel_cost'}
        if 'activity' in text or 'activities' in text or 'things to do' in text: return {'tool_to_call': 'recommend_local_activities'}
        if 'random' in text or 'inspire' in text: return {'tool_to_call': 'get_random_destination'}
    return {'tool_to_call': None}

router_node({'messages':[HumanMessage(content='Show me weather for Tokyo')]})

{'tool_to_call': 'get_weather_info'}

### 🔗 Tool Execution Node
Executes the selected tool (if any) and appends ToolMessage + aggregates structured data.

In [7]:
# Map tool name -> callable for execution
tool_map = {t.name: t for t in TOOLS}

def tool_node(state: AgentState):
    state = ensure_state(state)
    decision = state.get('tool_to_call')
    if not decision:
        return state  # no-op
    tool_fn = tool_map.get(decision)
    human = next((m for m in reversed(state['messages']) if isinstance(m, HumanMessage)), None)
    args = {}
    prompt = human.content if human else ''
    # Naive argument extraction; in production you'd parse with an LLM call or schema extraction
    if decision == 'get_weather_info':
        # assume 'in <place>' pattern fallback
        parts = prompt.split()
        loc = parts[-1].strip('?.') if parts else 'Tokyo'
        args = {'location': loc}
    elif decision == 'calculate_travel_cost':
        args = {'destination': 'Paris', 'days': 3, 'style': 'standard'}
    elif decision == 'recommend_local_activities':
        args = {'destination': 'Tokyo', 'interests': ['art','food'], 'max_results': 3}
    elif decision == 'get_random_destination':
        args = {}
    try:
        result = tool_fn.invoke(args) if args else tool_fn.invoke({})
    except Exception as e:
        result = {'error': str(e), 'tool': decision}
    # Append tool message
    state['messages'].append(ToolMessage(name=decision, content=json.dumps(result), tool_call_id=decision))
    # Aggregate
    state['aggregated'][decision] = result
    state['last_tool'] = decision
    return state

tool_node({'messages':[HumanMessage(content='Weather for Berlin')], 'tool_to_call':'get_weather_info'})['aggregated']

{'get_weather_info': {'location': 'Berlin',
  'forecast': 'sunny',
  'temp_c': 16.0}}

### 💬 Response (LLM) Node
Summarizes the conversation + any aggregated tool data into a final assistant message.

In [8]:
def llm_response_node(state: AgentState):
    state = ensure_state(state)
    aggregated = state.get('aggregated', {})
    user_msg = next((m for m in reversed(state['messages']) if isinstance(m, HumanMessage)), None)
    prompt_parts = [
        'You are a concise helpful travel assistant. Use any tool results provided.',
        f'User request: {user_msg.content if user_msg else ''}',
        'Tool data JSON:' + json.dumps(aggregated) if aggregated else 'No tool data.'
    ]
    ai = llm.invoke(prompt_parts)
    state['messages'].append(AIMessage(content=ai.content))
    return state

llm_response_node({'messages':[HumanMessage(content='Suggest activities'), ToolMessage(name='recommend_local_activities', content='[]', tool_call_id='test')]})['messages'][-1].content[:60]

'Sure! Here are some activities to consider based on common t'

### 🧵 Building the Graph
We add nodes and conditional edges for tool flow vs direct response.

In [10]:
graph = StateGraph(AgentState)
graph.add_node('router', router_node)
graph.add_node('tool', tool_node)
graph.add_node('respond', llm_response_node)
graph.set_entry_point('router')

def route_after_router(state: AgentState):
    return 'tool' if state.get('tool_to_call') else 'respond'

graph.add_conditional_edges('router', route_after_router, {'tool':'tool','respond':'respond'})
graph.add_edge('tool','respond')
graph.add_edge('respond', END)

memory = MemorySaver()
app = graph.compile(checkpointer=memory)


### ▶️ Running Sample Interactions

In [12]:
def run_query(text: str, thread_id: str):
    events = app.stream({'messages': [HumanMessage(content=text)]}, config={'configurable': {'thread_id': thread_id}})
    final_ai = None
    for ev in events:
        pass  # could introspect intermediate states here
    # Retrieve final state from memory
    state = memory.get({'configurable': {'thread_id': thread_id}})
    if state and 'messages' in state:
        for m in state['messages'][::-1]:
            if isinstance(m, AIMessage):
                final_ai = m; break
    return final_ai.content if final_ai else '(no response)'

print(run_query('Give me a random inspiration', 'sess1')[:200])
print(run_query('What is the weather in Tokyo today?', 'sess1')[:200])
print(run_query('Estimate cost for 5 day luxury trip to Paris', 'sess1')[:200])
print(run_query('Suggest art and food activities in Barcelona', 'sess1')[:200])

(no response)
(no response)
(no response)
(no response)
(no response)
(no response)
(no response)


### ✅ Good vs ❌ Bad Tool Design
**Good**: small, typed, single purpose, structured return.
**Bad**: overlapping, vague, multi-purpose.

In [13]:
# ❌ Bad examples (for illustration only – DO NOT REGISTER)
def random_place():  # duplicate of get_random_destination
    return get_random_destination.invoke({})

def getData(destination, days):  # vague, mixed concerns, no typing or validation
    return {
        'weather': 'Nice',
        'cost': days * 200,
        'activities': ['Something'],
        'raw': 'blob'
    }
print('Bad examples defined (not used).')

Bad examples defined (not used).


### 🔚 Next Steps
- Replace stub weather + cost with real APIs
- Add retrieval (RAG) for destination knowledge
- Introduce parallel tool execution branch
- Add evaluation harness for tool accuracy

> Tip: Convert heuristic arg extraction to structured function calling using LLM function schemas for robustness.